<a href="https://colab.research.google.com/github/patilhumesh88-spec/free/blob/main/LANGCHAIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
!pip install -qU langchain langchain-openai langchain-community duckduckgo-search

import os
from google.colab import userdata
from langchain_openai import ChatOpenAI

os.environ["OPENROUTER_API_KEY"] = userdata.get("X_AI_API_KEYS")

model = ChatOpenAI(
    model="qwen/qwen3-8b",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

response = model.invoke("What is the color of sky?")
print(response.content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 1.3 MB/s eta 0:00:00
The color of the sky is primarily **blue** during the day, but it can vary depending on atmospheric conditions and time of day. Here's a breakdown:

1. **Blue Sky (Daytime):**  
   - Sunlight scatters more efficiently in the **shorter wavelengths** (blue/violet) due to **Rayleigh scattering** by molecules and small particles in the atmosphere.  
   - While violet light is scattered even more than blue, our eyes are less sensitive to violet, and some of it is absorbed by the upper atmosphere. This combination makes the sky appear **blue** to us.

2. **Sunset/Sunrise (Red/Orange):**  
   - During these times, sunlight travels through more of the atmosphere, scattering out most blue light. The remaining light (longer wavelengths like red and orange) dominates, creating warm hues.

3. **Cloudy Days:**  
   - Clouds scatter all wavelengths of light equally, making the sky appear **white** or **gray**.

4. **Othe

In [30]:
from langchain_core.messages import SystemMessage,AIMessage,HumanMessage
LEET_tutor =SystemMessage(content ="""Give a concise but useful LeetCode analysis.
Do not provide long explanations.
Keep each field focused and short.
Return only the requested structured fields.

LeetCode Problem:
Two Sum

Given an array of integers nums and an integer target, return indices
of the two numbers such that they add up to target.
"""
)

LEARNER_prompt = HumanMessage(content="hey can you explain me the 169. Majority Element leet code problem  ")
full_prompt = [LEET_tutor,LEARNER_prompt]
response = model.invoke(full_prompt)
print(response.content)

{
  "Problem": "Majority Element",
  "Description": "Find the element appearing more than n/2 times in an array.",
  "Approach": "Boyer-Moore Voting Algorithm (O(n) time, O(1) space).",
  "Code": "Initialize candidate and count. Iterate, update candidate and count. If count=0, set candidate to current element.",
  "Complexity": "Time: O(n), Space: O(1)",
  "Key Insight": "Majority element's count dominates during iteration."
}


In [31]:
print(response.content)

{
  "Problem": "Majority Element",
  "Description": "Find the element appearing more than n/2 times in an array.",
  "Approach": "Boyer-Moore Voting Algorithm (O(n) time, O(1) space).",
  "Code": "Initialize candidate and count. Iterate, update candidate and count. If count=0, set candidate to current element.",
  "Complexity": "Time: O(n), Space: O(1)",
  "Key Insight": "Majority element's count dominates during iteration."
}


NOW we will integrate PYDANTIC (A python library) To get a structured output with a visualization of algorithm we will use in DSA

In [32]:
from pydantic import BaseModel,Field
from typing import List

class LEETCODER(BaseModel):
 """schema for the structured response in deatiled way and less feel of Wikipedia """
 conceptcovered : str=Field(description="concept covered in the problem")
 vidlink:List[str]=Field(description="List of the link to the video available online for reference ")
 hint:str=Field(description="hint to solve the problem")
 companyask:List[str]=Field(description="List of company asking the question")
 refque:List[str]=Field(description="List of reference question")


In [33]:
structured_assistant=model.with_structured_output(LEETCODER)
response=structured_assistant.invoke(full_prompt)
print(response)

conceptcovered='Hash maps, Boyer-Moore Voting Algorithm, Sorting' vidlink=['https://www.youtube.com/watch?v=J7Ih1k8uBKY'] hint='Use Boyer-Moore for O(n) time and O(1) space. Or sort and check middle element.' companyask=['Find the majority element in an array (appears > n/2 times).'] refque=['Hash map to count frequencies', 'Boyer-Moore Voting Algorithm', 'Sorting and checking middle element']


Managing the memory without agents

In [34]:
from langchain_core.messages import SystemMessage,AIMessage,HumanMessage

full_prompt = [LEET_tutor]
human_question =HumanMessage(content= "MY name is humesh")
full_prompt.append(human_question)
response = model.invoke(full_prompt)
full_prompt.append(response)
print(response.content)


full_prompt = [LEET_tutor]
human_question = HumanMessage(content="What is my name?")
full_prompt.append(human_question)
response = model.invoke(full_prompt)
full_prompt.append(response)
print(response.content)

{
  "Problem": "Two Sum",
  "Difficulty": "Medium",
  "Key Insight": "Use a hash map to track numbers and their indices for O(1) lookup.",
  "Time Complexity": "O(n)",
  "Space Complexity": "O(n)",
  "Code Example": "def two_sum(nums, target):\n    num_map = {}\n    for i, num in enumerate(nums):\n        complement = target - num\n        if complement in num_map:\n            return [num_map[complement], i]\n        num_map[num] = i\n    return []"
}
{"error": "Name not provided or accessible"}


In [35]:
full_prompt

[SystemMessage(content='Give a concise but useful LeetCode analysis.\nDo not provide long explanations.\nKeep each field focused and short.\nReturn only the requested structured fields.\n\nLeetCode Problem:\nTwo Sum\n\nGiven an array of integers nums and an integer target, return indices\nof the two numbers such that they add up to target.\n', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='{"error": "Name not provided or accessible"}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 259, 'prompt_tokens': 80, 'total_tokens': 339, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 244, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0.000127205, 'is_byok': False, 

USING AGENTS IN LANGCHAIN


In [36]:
from langchain_core import messages
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
mem_saver=InMemorySaver()
sum_middlewave=SummarizationMiddleware(model=model,trigger={"messages":10},keep_recent=3)

In [37]:
Leetut= create_agent (model=model,system_prompt=LEET_tutor,checkpointer=mem_saver,middleware=[sum_middlewave])
DSA_config={"configurable":{"thread_id":"dsa_sol"}}
response=Leetut.invoke({"messages":[{"role":"user","content":"HI I am humesh and I am here to learn dsa"}]},config=DSA_config)
print(response)

{'messages': [HumanMessage(content='HI I am humesh and I am here to learn dsa', additional_kwargs={}, response_metadata={}, id='9cfd03e2-3374-46b5-bad7-6f72160e1b81'), AIMessage(content='{\n  "Problem": "Two Sum",\n  "Difficulty": "Easy",\n  "Key Concepts": ["Hash Map", "Single Pass", "Complement Check"],\n  "Optimal Solution": "Use a hash map to track seen numbers and their indices. For each number, check if its complement (target - num) exists in the map.",\n  "Time Complexity": "O(n)",\n  "Space Complexity": "O(n)",\n  "Edge Cases": ["Duplicates", "Target sum with same element used twice"]\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 474, 'prompt_tokens': 88, 'total_tokens': 562, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 362, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0,

In [38]:
response=Leetut.invoke({"messages":[{"role":"user","content":"HI I am humesh and I am here to learn dsa"}]},config=DSA_config,Checkpointer=mem_saver)
print(response)

{'messages': [HumanMessage(content='HI I am humesh and I am here to learn dsa', additional_kwargs={}, response_metadata={}, id='9cfd03e2-3374-46b5-bad7-6f72160e1b81'), AIMessage(content='{\n  "Problem": "Two Sum",\n  "Difficulty": "Easy",\n  "Key Concepts": ["Hash Map", "Single Pass", "Complement Check"],\n  "Optimal Solution": "Use a hash map to track seen numbers and their indices. For each number, check if its complement (target - num) exists in the map.",\n  "Time Complexity": "O(n)",\n  "Space Complexity": "O(n)",\n  "Edge Cases": ["Duplicates", "Target sum with same element used twice"]\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 474, 'prompt_tokens': 88, 'total_tokens': 562, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 362, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0,

In [39]:
response=Leetut.invoke({"messages":[{"role":"user","content":"What is my name? and what i want to learn?"}]},config=DSA_config,Checkpointer=mem_saver)
print(response)

{'messages': [HumanMessage(content='HI I am humesh and I am here to learn dsa', additional_kwargs={}, response_metadata={}, id='9cfd03e2-3374-46b5-bad7-6f72160e1b81'), AIMessage(content='{\n  "Problem": "Two Sum",\n  "Difficulty": "Easy",\n  "Key Concepts": ["Hash Map", "Single Pass", "Complement Check"],\n  "Optimal Solution": "Use a hash map to track seen numbers and their indices. For each number, check if its complement (target - num) exists in the map.",\n  "Time Complexity": "O(n)",\n  "Space Complexity": "O(n)",\n  "Edge Cases": ["Duplicates", "Target sum with same element used twice"]\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 474, 'prompt_tokens': 88, 'total_tokens': 562, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 362, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0,

Equiping Tools


In [40]:
from langchain_core.tools import tool
@tool
def next_que (question:str) -> str:
  """
  this will help you to tell and suggest which leetcode problem
  you should solve next by checking onto the current one you are
  and also analyzing the past questions also
  """
  return"Observing the current and previous queestion pattern you should solve "

In [41]:
!pip install -U ddgs
from langchain_community.tools import DuckDuckGoSearchRun
web_search_tool=DuckDuckGoSearchRun()



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 698.8 kB/s eta 0:00:00


In [42]:
from langchain_core import messages
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
mem_saver=InMemorySaver()
sum_middlewave=SummarizationMiddleware(model=model,trigger={"messages":10},keep_recent=3)
tool_box=[next_que,web_search_tool]
Leetut= create_agent (model=model,system_prompt=LEET_tutor,checkpointer=mem_saver,middleware=[sum_middlewave],tools=tool_box)
DSA_config={"configurable":{"thread_id":"dsa_sol"}}
response=Leetut.invoke({"messages":[{"role":"user","content":"HI I am humesh and I am here to learn dsa"}]},config=DSA_config)
print(response)


{'messages': [HumanMessage(content='HI I am humesh and I am here to learn dsa', additional_kwargs={}, response_metadata={}, id='29e9b89c-a37c-4749-8908-838a57011b25'), AIMessage(content='{"problem": "Two Sum", "difficulty": "Easy", "category": "Array/Hash Map", "optimal_time": "O(n)", "approach": "Use a hash map to store values and their indices, checking for complement values as you iterate.", "next_problem": "Next Problem: Medium - Longest Substring Without Repeating Characters"}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 332, 'prompt_tokens': 338, 'total_tokens': 670, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 253, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0.000190606, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000190606

UI ADD-UP

In [44]:
import gradio as gd
def chat_with_tutor(user_message,history):

 response=Leetut.invoke({"messages":[{"role":"user","content":user_message}]},config=DSA_config)
 return response ["messages"][-1].content

demo=gd.ChatInterface(chat_with_tutor)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a33fe4aca828e8a638.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
